# Lattice (spatially fractionated) radiotherapy in pyCERR

An end-to-end **fill-in-the-blanks** workflow for **lattice radiotherapy (LRT / 3-D GRID / SFRT)**:

> DICOM (CT + RTSTRUCT) → pyCERR → build a lattice of high-dose **vertices**
> (peaks) inside the GTV → fluence optimization (**pyRadPlan**) that drives the
> peaks high while capping the surrounding **valley** → dose back in `planC` →
> peak-to-valley DVH check.

Lattice radiotherapy delivers a deliberately **heterogeneous** dose: small
spheres (≈10–15 mm) on a regular 20–50 mm lattice receive a high boost, while
the tissue between them gets a much lower dose. The peak-to-valley dose ratio
is the defining feature of the technique.

Edit the **CONFIG** cell (all placeholders are marked `# <-- EDIT`), then
Run-All. The notebook ships **no PHI** — you supply your own de-identified
DICOM.

**Modules:** `cerr.imrtp.lattice` (target creation + optimization wiring) and
`cerr.imrtp.pyradplan_bridge` (dose engine).

**Requires:** `pip install pyRadPlan` (0.4.x) in the `pycerr` environment.

## 1. CONFIG — plug in your dataset

Set the DICOM directory and the GTV name as it appears in your RTSTRUCT, plus
the lattice geometry and the peak/valley prescription.

In [ ]:
# ----------------------------- CONFIG ------------------------------------
DICOM_DIR   = r"C:/path/to/your/DICOM/folder"   # <-- EDIT: de-identified CT + RTSTRUCT
SCAN_NUM    = 0            # <-- EDIT: index of the planning CT in planC.scan
TARGET_NAME = "GTV"       # <-- EDIT: gross-tumor structure name in the RTSTRUCT
BODY_NAME   = "BODY"      # <-- EDIT: external/BODY structure name (calc box)

# --- Lattice geometry (clinical units: mm) ---
SPHERE_DIAMETER_MM = 15.0     # <-- EDIT: vertex sphere diameter (typ. 10-15 mm)
LATTICE_SPACING_MM = 30.0     # <-- EDIT: vertex centre-to-centre spacing (typ. 20-50 mm)
LATTICE_TYPE       = "bcc"   # <-- EDIT: 'bcc' (body-centred) or 'sc' (simple cubic)
INNER_MARGIN_MM    = 8.0      # <-- EDIT: inset of vertices from the GTV surface

# --- Prescription ---
PEAK_DOSE_GY     = 15.0       # <-- EDIT: boost dose at the vertices
VALLEY_MAX_GY    = None       # <-- EDIT: valley cap (None -> 30% of PEAK_DOSE_GY)
NUM_FRACTIONS    = 1          # <-- EDIT: single-fraction SFRT by default

# --- Beam geometry (static IMRT fields; VMAT arcs are NOT reproduced) ---
# Set GANTRY_ANGLES = None to read treatment beams from planC.beams[0].
GANTRY_ANGLES = [0.0, 51.0, 102.0, 153.0, 204.0, 255.0, 306.0]   # <-- EDIT

# --- OAR objectives: {structure_name: overdose_limit_gy} ---
OAR_OBJECTIVES = {}          # <-- EDIT, e.g. {"SpinalCord": 6.0, "Lung": 5.0}

# --- Beamlet / dose-grid resolution (mm) ---
BIXEL_WIDTH_MM = 5.0
DOSE_GRID_MM   = 4.0
# -------------------------------------------------------------------------

## 2. Load the DICOM dataset

`loadDcmDir` reads the CT scan and the RTSTRUCT (and RTPLAN/RTDOSE if present)
into a single `planC` container.

In [ ]:
import time
import numpy as np
from cerr import plan_container as pc
from cerr.imrtp import lattice as lat
from cerr.imrtp import pyradplan_bridge as prp

t0 = time.time()
planC = pc.loadDcmDir(DICOM_DIR)
print("loaded in %.1fs" % (time.time() - t0))
print("scans:", [(i, s.scanType, tuple(s.getScanSize()))
                 for i, s in enumerate(planC.scan)])
print("structures:", [s.structureName for s in planC.structure])
print("doses:", len(planC.dose), "| beams:", len(planC.beams))


def sidx(name):
    """Index of a structure by name (raises with the available names)."""
    hits = [i for i, s in enumerate(planC.structure)
            if s.structureName == name]
    if not hits:
        raise ValueError("structure %r not found; available: %s"
                         % (name, [s.structureName for s in planC.structure]))
    return hits[0]


gtvNum = sidx(TARGET_NAME)
print("GTV is structure index", gtvNum)

## 3. Build the lattice of vertices

`createLattice` places a regular lattice of spheres inside the GTV (inset by
`INNER_MARGIN_MM` so no sphere protrudes past the surface) and appends two new
structures to `planC`:

- **`<GTV>_lattice_peaks`** — the union of the high-dose vertices,
- **`<GTV>_lattice_valley`** — the GTV minus the peaks.

`verticesXYZ` returns the vertex centres (pyCERR cm coordinates).

In [ ]:
planC, verticesXYZ = lat.createLattice(
    planC, gtvNum,
    sphereDiameter=SPHERE_DIAMETER_MM,
    latticeSpacing=LATTICE_SPACING_MM,
    latticeType=LATTICE_TYPE,
    innerMargin=INNER_MARGIN_MM,
    units="mm",
    addValley=True,
    addIndividualSpheres=False)

peakNum   = sidx("%s_lattice_peaks" % TARGET_NAME)
valleyNum = sidx("%s_lattice_valley" % TARGET_NAME)
print("placed %d vertices -> peaks[%d], valley[%d]"
      % (len(verticesXYZ), peakNum, valleyNum))

### Visualize the lattice on the CT

Overlay the peaks (red) and valley (blue) on the CT slice that contains the
most vertices.

In [ ]:
import matplotlib.pyplot as plt
import cerr.contour.rasterseg as rs

scan3M   = planC.scan[SCAN_NUM].getScanArray()
peak3M   = rs.getStrMask(peakNum, planC)
valley3M = rs.getStrMask(valleyNum, planC)

# slice with the largest peak area
k = int(np.argmax(peak3M.sum(axis=(0, 1))))
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(scan3M[:, :, k], cmap="gray")
ax.imshow(np.ma.masked_where(~valley3M[:, :, k], valley3M[:, :, k]),
          cmap="Blues", alpha=0.35)
ax.imshow(np.ma.masked_where(~peak3M[:, :, k], peak3M[:, :, k]),
          cmap="autumn", alpha=0.8)
ax.set_title("lattice vertices (red) in valley (blue), slice %d" % k)
ax.axis("off")
plt.show()

## 4. Optimize the lattice boost and import the dose

`optimizeLatticeBoost` wires the peak/valley structures into a fluence-optimized
plan through the pyRadPlan bridge in one call:

1. drives the **peaks** to `PEAK_DOSE_GY` (a `SquaredDeviation` target),
2. caps the **valley** at `VALLEY_MAX_GY` (default 30% of the peak dose),
3. enforces a **GTV** minimum coverage,
4. adds any **OAR** overdose objectives,

then computes the beamlet dose-influence matrix, optimizes the fluence, and
appends the dose to `planC.dose`.

The isocenter defaults to the GTV centroid; beam geometry comes from
`GANTRY_ANGLES` (or `planC.beams[0]` when it is `None`).

In [ ]:
# Build OAR objectives {structNum: [Objective, ...]} from the config names.
oar_obj = {sidx(nm): [prp.squaredOverdosing(gy)]
           for nm, gy in OAR_OBJECTIVES.items()}

# Isocenter = GTV centroid in DICOM mm.
iso = prp.targetCentroidMm(planC, [gtvNum])

t0 = time.time()
w, doseNum, planC = lat.optimizeLatticeBoost(
    planC, peakNum,
    valleyStructNum=valleyNum,
    gtvStructNum=gtvNum,
    scanNum=SCAN_NUM,
    gantryAngles=GANTRY_ANGLES,
    isoCenter=iso,
    peakDose=PEAK_DOSE_GY,
    valleyMaxDose=VALLEY_MAX_GY,
    oarObjectives=oar_obj,
    numOfFractions=NUM_FRACTIONS,
    bixelWidth=BIXEL_WIDTH_MM,
    doseGridResolution={"x": DOSE_GRID_MM, "y": DOSE_GRID_MM, "z": DOSE_GRID_MM},
    fractionGroupID="lattice")
print("optimized %d beamlet weights in %.1fs -> planC.dose[%d]"
      % (w.size, time.time() - t0, doseNum))

## 5. Peak-to-valley check

The hallmark of a lattice plan is a high peak dose with a much lower valley
dose. We report mean doses in the peaks and valley and their ratio, then
overlay the dose on the CT.

In [ ]:
dose3M = planC.dose[doseNum].doseArray
peakMean   = float(dose3M[peak3M].mean())
valleyMean = float(dose3M[valley3M].mean())
print("peak   mean dose: %6.2f Gy" % peakMean)
print("valley mean dose: %6.2f Gy" % valleyMean)
print("peak-to-valley ratio: %.2f" % (peakMean / max(valleyMean, 1e-6)))

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(scan3M[:, :, k], cmap="gray")
m = ax.imshow(np.ma.masked_less(dose3M[:, :, k], 0.1 * dose3M.max()),
              cmap="jet", alpha=0.5)
plt.colorbar(m, ax=ax, label="Gy", fraction=0.046)
ax.set_title("lattice boost dose, slice %d  (max=%.1f Gy)" % (k, dose3M.max()))
ax.axis("off")
plt.show()

### DVH of peaks vs valley

In [ ]:
from cerr import dvh

fig, ax = plt.subplots(figsize=(6, 4))
for structNum, name, color in [(peakNum, "peaks", "tab:red"),
                               (valleyNum, "valley", "tab:blue"),
                               (gtvNum, "GTV", "tab:green")]:
    d, v, _ = dvh.getDVH(structNum, doseNum, planC)
    d = np.asarray(d); v = np.asarray(v)
    o = np.argsort(d); d, v = d[o], v[o]
    cum = np.cumsum(v[::-1])[::-1]
    ax.plot(d, 100.0 * cum / cum.max(), label=name, color=color)
ax.set_xlabel("Dose (Gy)"); ax.set_ylabel("Volume (%)")
ax.set_title("Lattice boost DVH"); ax.legend(); ax.grid(alpha=0.3)
plt.show()

## Notes & caveats

- **Units.** `createLattice` takes clinical **mm** by default (`units="mm"`);
  pass `units="cm"` to match pyCERR's native grid. `verticesXYZ` is returned in
  pyCERR cm coordinates.
- **Lattice geometry.** `"bcc"` (body-centred cubic) packs a body-centred point
  into each cell, giving more vertices than `"sc"` for the same spacing. Use
  `addIndividualSpheres=True` to also get one structure per vertex (e.g. for
  per-vertex QA).
- **Peak-to-valley ratio** is driven by `PEAK_DOSE_GY` vs `VALLEY_MAX_GY` and
  the objective priorities. Tighten spacing, add fields, or raise the valley
  priority to sharpen the gradient.
- **VMAT arcs are not reproduced** — this is a static-field IMRT re-plan.
- **Orientation & grids** are handled automatically (LPS reorientation inside
  the bridge; dose resampled back onto the scan grid on import).
- **No PHI** ships with this notebook — point `DICOM_DIR` at your own
  de-identified dataset.